# Vault Auto-Unseal with AWS KMS — Kubernetes Web Identity/OIDC

This notebook configures Vault auto-unseal without storing AWS access keys in Kubernetes. A projected Kubernetes ServiceAccount token is exchanged through `sts:AssumeRoleWithWebIdentity`; the AWS SDK refreshes the temporary credentials from the rotated token file.

```text
Vault pod → projected SA JWT → AWS STS → vault-kms-unseal-oidc role → AWS KMS
```

## Prerequisite

The Kubernetes issuer must use HTTPS, expose its API through ngrok, and be registered as an IAM OIDC provider. The local AWS credentials used below provision IAM/KMS only and are never mounted into Vault.

> Keep the Jupyter kernel and the ngrok process running while Vault uses Web Identity. This ngrok arrangement is for a lab; production requires a durable OIDC endpoint.

# How the Web Identity federation works

## Version support

The AWS KMS seal accepts `web_identity_token_file`, `role_arn`, and `role_session_name` starting with **Vault 1.8.4**. Vault 1.8.4 includes `go-kms-wrapping v0.6.7`, whose AWS wrapper contains these parameters. This notebook uses and has been verified with **Vault Enterprise 2.1.0**.


References: [Vault 1.8.4 dependencies](https://github.com/hashicorp/vault/blob/v1.8.4/go.mod), [AWS wrapper v0.6.7](https://github.com/hashicorp/go-kms-wrapping/blob/v0.6.7/wrappers/awskms/awskms.go), and [Vault AWS KMS seal configuration](https://developer.hashicorp.com/vault/docs/configuration/seal/awskms).

## Identity flow

```text
Kubernetes API server
  signs a ServiceAccount JWT
             │
             ▼
Vault pod: /var/run/secrets/aws/token
             │  sts:AssumeRoleWithWebIdentity
             ▼
AWS STS validates iss + signature + aud + sub + expiry
             │
             ▼
Temporary role credentials
             │
             ▼
AWS KMS DescribeKey / Encrypt / Decrypt
```

This is a federation between the Kubernetes workload identity and AWS IAM. Vault is the workload consuming the federated identity; Vault's own authentication methods are not involved. No permanent AWS access key is stored in the pod.

## Minikube API server configuration

Minikube starts the Kubernetes API server with:

```bash
--extra-config=apiserver.service-account-issuer=https://<domain>.ngrok.app
--extra-config=apiserver.service-account-jwks-uri=https://<domain>.ngrok.app/openid/v1/jwks
```

`service-account-issuer` sets the `iss` claim included in every projected ServiceAccount token. It must exactly match the URL registered in the AWS IAM OIDC provider. Because the issuer becomes part of the signed token and IAM trust policy, the ngrok domain must remain stable.

`service-account-jwks-uri` tells the discovery document where AWS can download the Kubernetes public signing keys. Kubernetes keeps the private signing key; the JWKS endpoint exposes only public keys.

## What ngrok exposes

Ngrok gives AWS an HTTPS route to the API server through `kubectl proxy`. The OIDC endpoints used by AWS are:

```text
https://<domain>.ngrok.app/.well-known/openid-configuration
https://<domain>.ngrok.app/openid/v1/jwks
```

The discovery document identifies the issuer and its `jwks_uri`. AWS fetches the JWKS to verify the JWT signature. This lab exposes an authenticated `kubectl proxy`, not just those paths, so the ngrok process must be stopped immediately after the demonstration. A production deployment should publish only durable OIDC discovery and JWKS endpoints.

> **Availability requirement:** the ngrok tunnel and `kubectl proxy` must remain running whenever a Vault pod starts or AWS STS needs to validate a newly rotated token. If the issuer or JWKS returns 404, Vault 2.1 reports `NoCredentialProviders: no valid providers in chain`. Existing nodes may remain operational, but a restarted node cannot auto-unseal until the same issuer URL is restored.

## TLS thumbprint versus JWT signing keys

Two independent trust mechanisms are involved:

- The **ngrok TLS certificate thumbprint** helps IAM validate the HTTPS OIDC endpoint.
- The **Kubernetes JWKS public keys** let STS validate the cryptographic signature of the ServiceAccount JWT.

The notebook reads the ngrok certificate chain, calculates its SHA-1 thumbprint, and supplies it when creating the IAM `OpenIDConnectProvider`. The TLS certificate is not the key that signs the Kubernetes JWT.

## Projected ServiceAccount token

The pod requests a bounded token through a projected volume:

```yaml
projected:
  sources:
    - serviceAccountToken:
        audience: sts.amazonaws.com
        expirationSeconds: 3600
        path: token
```

The resulting JWT contains claims similar to:

```json
{
  "iss": "https://<domain>.ngrok.app",
  "sub": "system:serviceaccount:vault:vault",
  "aud": ["sts.amazonaws.com"],
  "exp": 1788...
}
```

- `iss` identifies the Kubernetes issuer.
- `sub` identifies exactly the `vault` ServiceAccount in the `vault` namespace.
- `aud` restricts the token to AWS STS. The exact value is `sts.amazonaws.com`.
- `exp` limits the token lifetime.

`expirationSeconds: 3600` requests a one-hour token. The kubelet rotates it proactively, normally when it has reached about 80% of its lifetime (approximately 48 minutes here), and atomically updates the mounted file. This is separate from the lifetime of the AWS STS credentials. The AWS provider refreshes its STS credentials and rereads the projected Web Identity token when required. See [Kubernetes ServiceAccount token projection](https://kubernetes.io/docs/tasks/configure-pod-container/configure-service-account/).

## AWS IAM OIDC provider and role trust

The IAM `OpenIDConnectProvider` registers:

```text
Issuer URL: https://<domain>.ngrok.app
Client ID:  sts.amazonaws.com
Thumbprint: ngrok TLS chain fingerprint
```

Registration only establishes an identity provider; it grants no KMS permission. The role trust policy authorizes `sts:AssumeRoleWithWebIdentity` only when both claims match:

```json
{
  "<domain>.ngrok.app:sub": "system:serviceaccount:vault:vault",
  "<domain>.ngrok.app:aud": "sts.amazonaws.com"
}
```

STS verifies the issuer, JWT signature, audience, subject, and expiry. It then returns a temporary access key, secret key, session token, and expiration for `vault-kms-unseal-oidc`. The role's identity policy and the KMS key policy authorize `kms:DescribeKey`, `kms:Encrypt`, and `kms:Decrypt`.

## Vault seal configuration

```hcl
seal "awskms" {
  region                  = "eu-west-3"
  kms_key_id              = "<KMS key ID>"
  role_arn                = "arn:aws:iam::<account>:role/vault-kms-unseal-oidc"
  role_session_name       = "vault-auto-unseal-oidc"
  web_identity_token_file = "/var/run/secrets/aws/token"
}
```

- `region` locates the KMS service.
- `kms_key_id` identifies the key protecting Vault's barrier key.
- `role_arn` selects the federated IAM role.
- `role_session_name` labels the temporary session for AWS audit records; it is not another IAM role.
- `web_identity_token_file` points to the rotated Kubernetes JWT.

## Auto-unseal lifecycle

During the first `vault operator init`, Vault creates its barrier key, obtains STS credentials using the ServiceAccount JWT, asks KMS to encrypt the barrier key, and stores only the ciphertext in Raft. KMS does not encrypt every Vault secret; it protects the key Vault needs to open its storage barrier.

After a restart, each Vault node reads a valid projected JWT, calls `AssumeRoleWithWebIdentity`, receives temporary AWS credentials, and asks KMS to decrypt the stored barrier key. `vault-1` and `vault-2` use `retry_join` to enter the Raft cluster led by `vault-0`; they must never be initialized separately. Once joined, every node performs auto-unseal using the same federated role and KMS key.

## 0. Create an OIDC-enabled Minikube profile and expose its API with ngrok

Prerequisites: install ngrok, configure its authtoken once with `ngrok config add-authtoken`, and export a stable reserved domain, for example `NGROK_DOMAIN=vault-oidc-demo.ngrok.app`. A random URL cannot be used because the issuer is embedded in every ServiceAccount token and in the IAM trust policy. This lab exposes `kubectl proxy` through ngrok; stop both processes when the demo finishes.

In [15]:
import json
import os
import re
import shutil
import subprocess
import time
import urllib.request

NGROK_DOMAIN = os.environ.get('NGROK_DOMAIN', '').strip().removeprefix('https://').rstrip('/')
if not NGROK_DOMAIN:
    raise ValueError('Export a stable reserved domain: NGROK_DOMAIN=your-domain.ngrok.app')
if not shutil.which('ngrok'):
    raise RuntimeError('ngrok is not installed; install it and configure its authtoken first')

MINIKUBE_PROFILE = 'workshop-oidc'
OIDC_ISSUER_URL = f'https://{NGROK_DOMAIN}'
os.environ['OIDC_ISSUER_URL'] = OIDC_ISSUER_URL

subprocess.run([
    'minikube', 'start', '-p', MINIKUBE_PROFILE, '--force',
    f'--extra-config=apiserver.service-account-issuer={OIDC_ISSUER_URL}',
    f'--extra-config=apiserver.service-account-jwks-uri={OIDC_ISSUER_URL}/openid/v1/jwks'
], check=True)
subprocess.run(['kubectl', 'config', 'use-context', MINIKUBE_PROFILE], check=True)

# vfkit may advertise a non-responsive gateway DNS. Reuse the active macOS
# resolvers so pods can reach ngrok, STS, and KMS.
macos_dns = subprocess.check_output(['scutil', '--dns'], text=True)
dns_servers = []
for address in re.findall(r'nameserver\[\d+\] : ([0-9.]+)', macos_dns):
    if address not in dns_servers:
        dns_servers.append(address)
if not dns_servers:
    raise RuntimeError('No active IPv4 DNS resolvers found in macOS')
dns_command = 'sudo resolvectl dns eth0 ' + ' '.join(dns_servers[:3]) + '; sudo resolvectl domain eth0 ~.'
subprocess.run(['minikube', 'ssh', '-p', MINIKUBE_PROFILE, '--', dns_command], check=True)
subprocess.run(['kubectl', 'rollout', 'restart', 'deployment/coredns', '-n', 'kube-system'], check=True)
subprocess.run([
    'kubectl', 'rollout', 'status', 'deployment/coredns', '-n', 'kube-system', '--timeout=120s'
], check=True)

# Pull on macOS and load into the node, avoiding registry DNS problems inside
# the Minikube VM. The Podman image is reused when already present.
if not shutil.which('podman'):
    raise RuntimeError('Podman is required to preload the Vault Enterprise image')
vault_image = 'docker.io/hashicorp/vault-enterprise:2.1.0-ent'
image_archive = '/tmp/vault-enterprise-2.1.0-ent.tar'
subprocess.run(['podman', 'pull', vault_image], check=True)
subprocess.run(['podman', 'save', '--format', 'docker-archive', '-o', image_archive, vault_image], check=True)
subprocess.run(['minikube', 'image', 'load', '-p', MINIKUBE_PROFILE, image_archive], check=True)

# kubectl proxy authenticates against the API server. For this isolated lab,
# ngrok exposes it so AWS can read OIDC discovery and JWKS. Stop stale
# processes even when they were created by another Jupyter kernel.
if 'kubectl_proxy_process' in globals() and kubectl_proxy_process.poll() is None:
    kubectl_proxy_process.terminate()
if 'ngrok_process' in globals() and ngrok_process.poll() is None:
    ngrok_process.terminate()
subprocess.run(['pkill', '-f', rf'ngrok.*{re.escape(NGROK_DOMAIN)}'], check=False)
subprocess.run(['pkill', '-f', r'kubectl.*proxy.*--port=8001'], check=False)
time.sleep(2)
kubectl_proxy_process = subprocess.Popen([
    'kubectl', '--context', MINIKUBE_PROFILE, 'proxy',
    '--address=127.0.0.1', '--port=8001', '--accept-hosts=.*'
], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
time.sleep(2)
if kubectl_proxy_process.poll() is not None:
    raise RuntimeError(f'kubectl proxy stopped: {kubectl_proxy_process.stderr.read()}')

ngrok_process = subprocess.Popen(
    ['ngrok', 'http', '8001', f'--url={OIDC_ISSUER_URL}'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

discovery_url = f'{OIDC_ISSUER_URL}/.well-known/openid-configuration'
for attempt in range(1, 31):
    if ngrok_process.poll() is not None:
        raise RuntimeError(f'ngrok stopped: {ngrok_process.stderr.read()}')
    try:
        with urllib.request.urlopen(discovery_url, timeout=5) as response:
            discovery = json.load(response)
        break
    except Exception:
        if attempt == 30:
            raise
        time.sleep(2)

assert discovery['issuer'].rstrip('/') == OIDC_ISSUER_URL
assert discovery['jwks_uri'] == f'{OIDC_ISSUER_URL}/openid/v1/jwks'
with urllib.request.urlopen(discovery['jwks_uri'], timeout=10) as response:
    jwks = json.load(response)
assert jwks.get('keys'), 'The Kubernetes JWKS contains no signing keys'

# IAM accepts the SHA-1 thumbprint of the top certificate presented by the
# issuer. Supplying it keeps the notebook compatible with older API models.
tls_chain = subprocess.check_output(
    ['openssl', 's_client', '-showcerts', '-servername', NGROK_DOMAIN, '-connect', f'{NGROK_DOMAIN}:443'],
    input=b'', stderr=subprocess.DEVNULL
)
certificates = re.findall(
    b'-----BEGIN CERTIFICATE-----.*?-----END CERTIFICATE-----', tls_chain, re.DOTALL
)
if not certificates:
    raise RuntimeError('Unable to read the ngrok TLS certificate chain')
fingerprint_line = subprocess.check_output(
    ['openssl', 'x509', '-noout', '-fingerprint', '-sha1'], input=certificates[-1], text=False
).decode().strip()
os.environ['OIDC_THUMBPRINT'] = fingerprint_line.split('=', 1)[1].replace(':', '').lower()
print(f'✓ Minikube context: {MINIKUBE_PROFILE}')
print(f'✓ Public OIDC issuer: {OIDC_ISSUER_URL}')
print(f'✓ Published signing keys: {len(jwks["keys"])}')

* [workshop-oidc] minikube v1.38.1 on Darwin 26.5.2 (arm64)
* Automatically selected the vfkit driver. Other choices: ssh, podman (experimental)
* Starting "workshop-oidc" primary control-plane node in "workshop-oidc" cluster


! minikube skips various validations when --force is supplied; this may lead to unexpected behavior
! Starting v1.39.0, minikube will default to "containerd" container runtime. See #21973 for more info.
! Failing to connect to https://registry.k8s.io/ from inside the minikube VM
* To pull new external images, you may need to configure a proxy: https://minikube.sigs.k8s.io/docs/reference/networking/proxy/


  - apiserver.service-account-issuer=https://nondemanding-indistinguishably-claude.ngrok-free.dev
  - apiserver.service-account-jwks-uri=https://nondemanding-indistinguishably-claude.ngrok-free.dev/openid/v1/jwks
* Configuring bridge CNI (Container Networking Interface) ...
* Verifying Kubernetes components...
  - Using image gcr.io/k8s-minikube/storage-provisioner:v5
* Enabled addons: default-storageclass, storage-provisioner
* Done! kubectl is now configured to use "workshop-oidc" cluster and "default" namespace by default
Switched to context "workshop-oidc".
deployment.apps/coredns restarted
Waiting for deployment spec update to be observed...
Waiting for deployment spec update to be observed...
Waiting for deployment "coredns" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "coredns" rollout to finish: 0 of 1 updated replicas are available...
deployment "coredns" successfully rolled out


Trying to pull docker.io/hashicorp/vault-enterprise:2.1.0-ent...
Getting image source signatures
Copying blob sha256:77285bba34fedc7ccfb1126b25c8e917bcc9fd67bbfc5a59bccec3573f4f8a2f
Copying blob sha256:3d7fab01012f71587108701b746a285a24350b23d8a269a9811c43a427b02c73
Copying blob sha256:bd17f05d207095d9539606c4866a259037df8248c80ac9c72fc1c8a26bbe31e4
Copying blob sha256:01a74a68af83fa2cee3ecde7a46f4a62d86a280d1f1c84162ad9ef1e3b93a1c1
Copying blob sha256:06d3564fe731f978a0f110d12244779cf2ebf9a2b255ad13c93a7e36d77395d6
Copying blob sha256:e796369152ae2bcfc5a6770ec686c48258300b27adec12edb4c13f9ab41af2f5
Copying blob sha256:94f653e8747c42087381445705f51567ea47ed3b4ad1f7f08dc2e350af70aa19
Copying config sha256:c00ac898e23113506ed91a6d959290749759f7113783090a42441ac21ddf4ff5
Writing manifest to image destination


c00ac898e23113506ed91a6d959290749759f7113783090a42441ac21ddf4ff5
✓ Minikube context: workshop-oidc
✓ Public OIDC issuer: https://nondemanding-indistinguishably-claude.ngrok-free.dev
✓ Published signing keys: 1


## 1. Parameters and provisioning credentials

In [16]:
import csv
import json
import os
import time
from urllib.parse import urlparse

import boto3
from botocore.exceptions import ClientError

REGION = os.environ.get('AWS_REGION', 'eu-west-3')
NAMESPACE = os.environ.get('VAULT_K8S_NAMESPACE', 'vault')
SERVICE_ACCOUNT = os.environ.get('VAULT_SERVICE_ACCOUNT', 'vault')
ROLE_NAME = 'vault-kms-unseal-oidc'
KMS_ALIAS = 'alias/vault-auto-unseal-oidc'
OIDC_ISSUER_URL = os.environ.get('OIDC_ISSUER_URL', '').rstrip('/')
OIDC_THUMBPRINT = os.environ.get('OIDC_THUMBPRINT', '').replace(':', '').lower()

if not OIDC_ISSUER_URL:
    raise ValueError('Set OIDC_ISSUER_URL to the public HTTPS issuer for this Kubernetes cluster')
parsed_issuer = urlparse(OIDC_ISSUER_URL)
if parsed_issuer.scheme != 'https' or not parsed_issuer.netloc:
    raise ValueError('OIDC_ISSUER_URL must be a public HTTPS URL')
if parsed_issuer.hostname in {'kubernetes.default.svc', 'kubernetes.default.svc.cluster.local', 'localhost'}:
    raise ValueError('The internal Kubernetes issuer is not reachable by AWS STS')
OIDC_HOSTPATH = OIDC_ISSUER_URL.removeprefix('https://')

# Administrative credentials used only by this notebook to provision AWS.
with open('vault_test_accessKeys.csv', encoding='utf-8-sig') as csvfile:
    creds = next(csv.DictReader(csvfile))
os.environ['AWS_ACCESS_KEY_ID'] = creds['Access key ID'].strip()
os.environ['AWS_SECRET_ACCESS_KEY'] = creds['Secret access key'].strip()
os.environ.pop('AWS_SESSION_TOKEN', None)
os.environ.pop('AWS_SECURITY_TOKEN', None)
os.environ['AWS_REGION'] = REGION
os.environ['VAULT_K8S_NAMESPACE'] = NAMESPACE
os.environ['VAULT_SERVICE_ACCOUNT'] = SERVICE_ACCOUNT
print(f'OIDC issuer: {OIDC_ISSUER_URL}')
print(f'ServiceAccount subject: system:serviceaccount:{NAMESPACE}:{SERVICE_ACCOUNT}')

OIDC issuer: https://nondemanding-indistinguishably-claude.ngrok-free.dev
ServiceAccount subject: system:serviceaccount:vault:vault


## 2. Register the OIDC provider and create the KMS role

The trust policy requires both the exact ServiceAccount subject and the `sts.amazonaws.com` audience. The role receives the KMS permissions; no IAM user credentials are delivered to the pod.

In [17]:
iam = boto3.client('iam')
kms = boto3.client('kms', region_name=REGION)
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
role_arn = f'arn:aws:iam::{account_id}:role/{ROLE_NAME}'
subject = f'system:serviceaccount:{NAMESPACE}:{SERVICE_ACCOUNT}'

# Reuse the provider when the issuer is already registered.
provider_arn = None
for item in iam.list_open_id_connect_providers().get('OpenIDConnectProviderList', []):
    details = iam.get_open_id_connect_provider(OpenIDConnectProviderArn=item['Arn'])
    if details['Url'].rstrip('/') == OIDC_HOSTPATH:
        provider_arn = item['Arn']
        break

if provider_arn is None:
    create_args = {
        'Url': OIDC_ISSUER_URL,
        'ClientIDList': ['sts.amazonaws.com'],
        'Tags': [{'Key': 'Purpose', 'Value': 'vault-auto-unseal-oidc'}]
    }
    if OIDC_THUMBPRINT:
        create_args['ThumbprintList'] = [OIDC_THUMBPRINT]
    provider_arn = iam.create_open_id_connect_provider(**create_args)['OpenIDConnectProviderArn']
    print(f'✓ IAM OIDC provider created: {provider_arn}')
else:
    print(f'✓ IAM OIDC provider already exists: {provider_arn}')

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Federated': provider_arn},
        'Action': 'sts:AssumeRoleWithWebIdentity',
        'Condition': {
            'StringEquals': {
                f'{OIDC_HOSTPATH}:sub': subject,
                f'{OIDC_HOSTPATH}:aud': 'sts.amazonaws.com'
            }
        }
    }]
}
try:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Web Identity role for Vault AWS KMS auto-unseal',
        MaxSessionDuration=3600,
        Tags=[{'Key': 'Purpose', 'Value': 'vault-auto-unseal-oidc'}]
    )
    print(f'✓ IAM role created: {role_arn}')
    time.sleep(15)
except ClientError as exc:
    if exc.response['Error']['Code'] != 'EntityAlreadyExists':
        raise
    iam.update_assume_role_policy(RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust_policy))
    print(f'✓ IAM role trust policy updated: {role_arn}')

# Create a separate KMS key so this notebook cannot disturb the shared-creds demo.
try:
    key = kms.describe_key(KeyId=KMS_ALIAS)['KeyMetadata']
    kms_key_id, kms_key_arn = key['KeyId'], key['Arn']
    print(f'✓ KMS key already exists: {kms_key_id}')
except ClientError as exc:
    if exc.response['Error']['Code'] != 'NotFoundException':
        raise
    root_policy = {
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'EnableRootAccountFullAccess',
            'Effect': 'Allow',
            'Principal': {'AWS': f'arn:aws:iam::{account_id}:root'},
            'Action': 'kms:*',
            'Resource': '*'
        }]
    }
    key = kms.create_key(
        Description='Vault auto-unseal key using Kubernetes Web Identity',
        Policy=json.dumps(root_policy),
        Tags=[{'TagKey': 'Purpose', 'TagValue': 'vault-auto-unseal-oidc'}]
    )['KeyMetadata']
    kms_key_id, kms_key_arn = key['KeyId'], key['Arn']
    kms.create_alias(AliasName=KMS_ALIAS, TargetKeyId=kms_key_id)
    print(f'✓ KMS key created: {kms_key_id}')

role_kms_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Action': ['kms:DescribeKey', 'kms:Encrypt', 'kms:Decrypt'],
        'Resource': kms_key_arn
    }]
}
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName='vault-kms-auto-unseal-oidc',
    PolicyDocument=json.dumps(role_kms_policy)
)
key_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Sid': 'EnableRootAccountFullAccess',
            'Effect': 'Allow',
            'Principal': {'AWS': f'arn:aws:iam::{account_id}:root'},
            'Action': 'kms:*',
            'Resource': '*'
        },
        {
            'Sid': 'AllowVaultWebIdentityRole',
            'Effect': 'Allow',
            'Principal': {'AWS': role_arn},
            'Action': ['kms:DescribeKey', 'kms:Encrypt', 'kms:Decrypt'],
            'Resource': '*'
        }
    ]
}
for attempt in range(1, 7):
    try:
        kms.put_key_policy(KeyId=kms_key_id, PolicyName='default', Policy=json.dumps(key_policy))
        break
    except ClientError as exc:
        if exc.response['Error']['Code'] != 'MalformedPolicyDocumentException' or attempt == 6:
            raise
        time.sleep(10)

os.environ['KMS_KEY_ID'] = kms_key_id
os.environ['VAULT_AWS_ROLE_ARN'] = role_arn
print(f'✓ Role ARN: {role_arn}')
print(f'✓ KMS key ARN: {kms_key_arn}')

✓ IAM OIDC provider created: arn:aws:iam::467008425114:oidc-provider/nondemanding-indistinguishably-claude.ngrok-free.dev
✓ IAM role created: arn:aws:iam::467008425114:role/vault-kms-unseal-oidc
✓ KMS key created: 234eabb2-5c2e-46bc-9603-98f0f9f17b78
✓ Role ARN: arn:aws:iam::467008425114:role/vault-kms-unseal-oidc
✓ KMS key ARN: arn:aws:kms:eu-west-3:467008425114:key/234eabb2-5c2e-46bc-9603-98f0f9f17b78


## 3. ServiceAccount and projected Web Identity token

The audience and subject in the projected token must exactly match the IAM role trust policy.

In [18]:
%%bash
set -euo pipefail
kubectl create namespace "${VAULT_K8S_NAMESPACE}" --dry-run=client -o yaml | kubectl apply -f -
kubectl create serviceaccount "${VAULT_SERVICE_ACCOUNT}" \
  --namespace "${VAULT_K8S_NAMESPACE}" \
  --dry-run=client -o yaml | kubectl apply -f -
echo "✓ ServiceAccount ${VAULT_K8S_NAMESPACE}/${VAULT_SERVICE_ACCOUNT} ready"

namespace/vault created
serviceaccount/vault created
✓ ServiceAccount vault/vault ready


## 4. Create TLS and Enterprise license Secrets

For this isolated Minikube profile, create demo TLS material and load the local `vault.hclic` license file.

In [19]:
%%bash
set -euo pipefail
WORKDIR=${WORKDIR:-/tmp/vault-oidc}
mkdir -p "${WORKDIR}"
test -s vault.hclic || { echo "vault.hclic not found" >&2; exit 1; }
openssl req -x509 -nodes -newkey rsa:2048 -days 30 \
  -keyout "${WORKDIR}/vault.key" \
  -out "${WORKDIR}/vault.crt" \
  -subj '/CN=vault.vault.svc' \
  -addext 'subjectAltName=DNS:vault,DNS:vault.vault,DNS:vault.vault.svc,DNS:*.vault-internal,DNS:*.vault-internal.vault.svc,IP:127.0.0.1'
cp "${WORKDIR}/vault.crt" "${WORKDIR}/vault.ca"
kubectl create secret generic vault-ha-tls -n "${VAULT_K8S_NAMESPACE}" \
  --from-file=vault.key="${WORKDIR}/vault.key" \
  --from-file=vault.crt="${WORKDIR}/vault.crt" \
  --from-file=vault.ca="${WORKDIR}/vault.ca" \
  --dry-run=client -o yaml | kubectl apply -f -
kubectl create secret generic vault-ent-license -n "${VAULT_K8S_NAMESPACE}" \
  --from-file=license=vault.hclic \
  --dry-run=client -o yaml | kubectl apply -f -
echo '✓ TLS and Vault Enterprise license Secrets ready'

Generating a 2048 bit RSA private key
.........................+++++
............................................+++++
writing new private key to '/tmp/vault-oidc/vault.key'
-----


secret/vault-ha-tls created
secret/vault-ent-license created
✓ TLS and Vault Enterprise license Secrets ready


## 5. Vault Helm overrides

No AWS credential Secret is used.

In [20]:
%%bash
set -euo pipefail
WORKDIR=${WORKDIR:-/tmp/vault-oidc}
mkdir -p "${WORKDIR}"
cat > "${WORKDIR}/overrides-oidc.yaml" <<EOF
global:
  enabled: true
  tlsDisable: false

injector:
  enabled: false

server:
  image:
    repository: docker.io/hashicorp/vault-enterprise
    tag: 2.1.0-ent
  enterpriseLicense:
    secretName: vault-ent-license
  serviceAccount:
    create: false
    name: ${VAULT_SERVICE_ACCOUNT}
  extraEnvironmentVars:
    VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
    VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
    VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key
  volumes:
    - name: userconfig-vault-ha-tls
      secret:
        defaultMode: 420
        secretName: vault-ha-tls
    - name: aws-web-identity
      projected:
        defaultMode: 420
        sources:
          - serviceAccountToken:
              audience: sts.amazonaws.com
              expirationSeconds: 3600
              path: token
  volumeMounts:
    - name: userconfig-vault-ha-tls
      mountPath: /vault/userconfig/vault-ha-tls
      readOnly: true
    - name: aws-web-identity
      mountPath: /var/run/secrets/aws
      readOnly: true
  standalone:
    enabled: false
  affinity: ""
  ha:
    enabled: true
    replicas: 3
    raft:
      enabled: true
      setNodeId: true
      config: |
        ui = true
        listener "tcp" {
          address            = "[::]:8200"
          cluster_address    = "[::]:8201"
          tls_disable        = 0
          tls_cert_file      = "/vault/userconfig/vault-ha-tls/vault.crt"
          tls_key_file       = "/vault/userconfig/vault-ha-tls/vault.key"
          tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
        }
        storage "raft" {
          path = "/vault/data"
          retry_join {
            leader_api_addr       = "https://vault-0.vault-internal:8200"
            leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
            leader_tls_servername = "vault-0.vault-internal"
          }
        }
        seal "awskms" {
          region                  = "${AWS_REGION}"
          kms_key_id              = "${KMS_KEY_ID}"
          role_arn                = "${VAULT_AWS_ROLE_ARN}"
          role_session_name       = "vault-auto-unseal-oidc"
          web_identity_token_file = "/var/run/secrets/aws/token"
        }
        disable_mlock = true
        service_registration "kubernetes" {}
EOF
echo "✓ ${WORKDIR}/overrides-oidc.yaml written"

✓ /tmp/vault-oidc/overrides-oidc.yaml written


## 6. Deploy and verify

A successful startup proves `AssumeRoleWithWebIdentity` and KMS access. Inspecting the JWT claims verifies issuer, subject, and audience without printing the token itself.

In [21]:
%%bash
set -euo pipefail
WORKDIR=${WORKDIR:-/tmp/vault-oidc}
discovery_url="${OIDC_ISSUER_URL}/.well-known/openid-configuration"
jwks_url="${OIDC_ISSUER_URL}/openid/v1/jwks"
curl --fail --silent --show-error --max-time 15 "${discovery_url}" >/dev/null
curl --fail --silent --show-error --max-time 15 "${jwks_url}" >/dev/null
echo '✓ Public OIDC discovery and JWKS are reachable'
helm repo add hashicorp https://helm.releases.hashicorp.com --force-update
helm upgrade --install vault hashicorp/vault \
  --namespace "${VAULT_K8S_NAMESPACE}" \
  --values "${WORKDIR}/overrides-oidc.yaml"
# Helm may return before the StatefulSet has created vault-0.
for attempt in $(seq 1 60); do
  kubectl get pod vault-0 --namespace "${VAULT_K8S_NAMESPACE}" >/dev/null 2>&1 && break
  if [ "${attempt}" -eq 60 ]; then
    echo 'vault-0 was not created within 120 seconds' >&2
    kubectl get statefulset,pods,events --namespace "${VAULT_K8S_NAMESPACE}"
    exit 1
  fi
  sleep 2
done
kubectl wait pod/vault-0 --namespace "${VAULT_K8S_NAMESPACE}" \
  --for=jsonpath='{.status.phase}'=Running --timeout=180s
kubectl logs vault-0 --namespace "${VAULT_K8S_NAMESPACE}" --tail=80 | \
  grep -E -i 'seal|awskms|web.identity|sts|kms|error' || true

✓ Public OIDC discovery and JWKS are reachable
"hashicorp" has been added to your repositories
Release "vault" does not exist. Installing it now.
NAME: vault
LAST DEPLOYED: Wed Sep  2 12:43:24 2026
NAMESPACE: vault
STATUS: deployed
REVISION: 1
DESCRIPTION: Install complete
NOTES:
Thank you for installing HashiCorp Vault!

Now that you have deployed Vault, you should look over the docs on using
Vault with Kubernetes available here:

https://developer.hashicorp.com/vault/docs


Your release is named vault. To learn more about the release, try:

  $ helm status vault
  $ helm get manifest vault
pod/vault-0 condition met


In [22]:
import base64
import json
import subprocess

token = subprocess.check_output([
    'kubectl', 'exec', '-n', NAMESPACE, 'vault-0', '--',
    'cat', '/var/run/secrets/aws/token'
], text=True).strip()
payload = token.split('.')[1]
payload += '=' * (-len(payload) % 4)
claims = json.loads(base64.urlsafe_b64decode(payload))
safe_claims = {key: claims.get(key) for key in ('iss', 'sub', 'aud', 'exp')}
print(json.dumps(safe_claims, indent=2))
assert claims['iss'].rstrip('/') == OIDC_ISSUER_URL
assert claims['sub'] == f'system:serviceaccount:{NAMESPACE}:{SERVICE_ACCOUNT}'
assert 'sts.amazonaws.com' in ([claims['aud']] if isinstance(claims['aud'], str) else claims['aud'])
print('✓ Projected token claims match the IAM trust policy')

{
  "iss": "https://nondemanding-indistinguishably-claude.ngrok-free.dev",
  "sub": "system:serviceaccount:vault:vault",
  "aud": [
    "sts.amazonaws.com"
  ],
  "exp": 1788349404
}
✓ Projected token claims match the IAM trust policy


## 7. Initialize and prove auto-unseal

Initialization creates recovery material because the barrier key is protected by AWS KMS. Sensitive output is written to a mode-0600 file and is never printed. Recreating `vault-0` then proves that Web Identity can obtain fresh STS credentials and decrypt the stored key automatically.

In [23]:
%%bash
set -euo pipefail
WORKDIR=${WORKDIR:-/tmp/vault-oidc}
curl --fail --silent --show-error --max-time 15 \
  "${OIDC_ISSUER_URL}/.well-known/openid-configuration" >/dev/null
curl --fail --silent --show-error --max-time 15 \
  "${OIDC_ISSUER_URL}/openid/v1/jwks" >/dev/null
echo '✓ Public OIDC issuer is reachable before initialization/restart'
mkdir -p "${WORKDIR}"
umask 077
status_json=$(kubectl exec -n "${VAULT_K8S_NAMESPACE}" vault-0 -- vault status -format=json 2>/dev/null || true)
if echo "${status_json}" | grep -q '"initialized": false'; then
  kubectl exec -n "${VAULT_K8S_NAMESPACE}" vault-0 -- \
    vault operator init -format=json -recovery-shares=1 -recovery-threshold=1 \
    > "${WORKDIR}/vault-init-oidc.json"
  chmod 600 "${WORKDIR}/vault-init-oidc.json"
  echo "✓ Vault initialized; recovery material saved to ${WORKDIR}/vault-init-oidc.json"
else
  echo '✓ Vault was already initialized'
fi
kubectl delete pod vault-0 -n "${VAULT_K8S_NAMESPACE}"
for attempt in $(seq 1 60); do
  kubectl get pod vault-0 -n "${VAULT_K8S_NAMESPACE}" >/dev/null 2>&1 && break
  sleep 2
done
kubectl wait pod/vault-0 -n "${VAULT_K8S_NAMESPACE}" \
  --for=jsonpath='{.status.phase}'=Running --timeout=180s
for pod in vault-0 vault-1 vault-2; do
  joined=false
  for attempt in $(seq 1 90); do
    seal_status=$(kubectl exec -n "${VAULT_K8S_NAMESPACE}" "${pod}" -- \
      vault status -format=json 2>/dev/null || true)
    if echo "${seal_status}" | grep -q '"initialized": true' && \
       echo "${seal_status}" | grep -q '"sealed": false'; then
      echo "✓ ${pod} joined the Raft cluster and is unsealed"
      joined=true
      break
    fi
    sleep 2
  done
  if [ "${joined}" != true ]; then
    echo "${pod} did not join and auto-unseal within 180 seconds" >&2
    kubectl logs "${pod}" -n "${VAULT_K8S_NAMESPACE}" --tail=80 >&2
    exit 1
  fi
done
echo '✓ Three-node Vault Raft cluster is initialized and unsealed'

✓ Public OIDC issuer is reachable before initialization/restart
✓ Vault initialized; recovery material saved to /tmp/vault-oidc/vault-init-oidc.json
pod "vault-0" deleted from vault namespace
pod/vault-0 condition met
✓ vault-0 joined the Raft cluster and is unsealed
✓ vault-1 joined the Raft cluster and is unsealed
✓ vault-2 joined the Raft cluster and is unsealed
✓ Three-node Vault Raft cluster is initialized and unsealed


## 8. Complete local cleanup: Vault, Minikube, ngrok, and Podman

This removes only resources created by this notebook. It stops the public tunnel, uninstalls Vault, deletes the `vault` namespace and the `workshop-oidc` profile, removes the Vault Enterprise image from Podman, and deletes local temporary files including initialization material. The deletion of `/tmp/vault-oidc/vault-init-oidc.json` is irreversible.

In [24]:
import pathlib
import re
import shutil
import subprocess

for process_name in ('ngrok_process', 'kubectl_proxy_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)
        print(f'✓ Stopped {process_name}')

# Popen objects only exist in the kernel that created them. These scoped
# patterns also stop leftovers from an older kernel or interrupted run.
subprocess.run(['pkill', '-f', rf'ngrok.*{re.escape(NGROK_DOMAIN)}'], check=False)
subprocess.run(['pkill', '-f', r'kubectl.*proxy.*--port=8001'], check=False)
print('✓ Stale ngrok and kubectl proxy processes removed')

subprocess.run(['helm', 'uninstall', 'vault', '--namespace', NAMESPACE], check=False)
subprocess.run([
    'kubectl', 'delete', 'namespace', NAMESPACE, '--ignore-not-found=true', '--wait=true', '--timeout=180s'
], check=False)
print(f'✓ Vault release and namespace removed: {NAMESPACE}')

subprocess.run(['minikube', 'delete', '-p', MINIKUBE_PROFILE], check=False)
print(f'✓ Minikube profile removed: {MINIKUBE_PROFILE}')

vault_image = 'docker.io/hashicorp/vault-enterprise:2.1.0-ent'
subprocess.run(['podman', 'image', 'rm', '--force', vault_image], check=False)
print(f'✓ Demo image removed from Podman: {vault_image}')

pathlib.Path('/tmp/vault-enterprise-2.1.0-ent.tar').unlink(missing_ok=True)
shutil.rmtree('/tmp/vault-oidc', ignore_errors=True)
print('✓ Local archives, Helm overrides, TLS keys, and Vault initialization material removed')

✓ Stopped ngrok_process
✓ Stopped kubectl_proxy_process
✓ Stale ngrok and kubectl proxy processes removed
release "vault" uninstalled
namespace "vault" deleted
✓ Vault release and namespace removed: vault
* Deleting "workshop-oidc" in vfkit ...
* Removed all traces of the "workshop-oidc" cluster.
✓ Minikube profile removed: workshop-oidc
Untagged: docker.io/hashicorp/vault-enterprise:2.1.0-ent
Deleted: c00ac898e23113506ed91a6d959290749759f7113783090a42441ac21ddf4ff5
✓ Demo image removed from Podman: docker.io/hashicorp/vault-enterprise:2.1.0-ent
✓ Local archives, Helm overrides, TLS keys, and Vault initialization material removed


## 9. Complete AWS cleanup

Run only after the local cleanup. This removes the role and all of its policies, deletes the KMS alias, schedules the KMS key for deletion with AWS's minimum seven-day recovery window, and deletes the IAM OIDC provider created for ngrok. Scheduling is the most complete deletion AWS KMS permits; the key cannot be destroyed immediately.

In [25]:
iam = boto3.client('iam')
kms = boto3.client('kms', region_name=REGION)

try:
    for name in iam.list_role_policies(RoleName=ROLE_NAME)['PolicyNames']:
        iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=name)
        print(f'✓ Deleted inline role policy: {name}')
    for policy in iam.list_attached_role_policies(RoleName=ROLE_NAME)['AttachedPolicies']:
        iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy['PolicyArn'])
        print(f'✓ Detached managed role policy: {policy["PolicyArn"]}')
    iam.delete_role(RoleName=ROLE_NAME)
    print(f'✓ IAM role deleted: {ROLE_NAME}')
except ClientError as exc:
    if exc.response['Error']['Code'] != 'NoSuchEntity':
        raise
    print(f'✓ IAM role already absent: {ROLE_NAME}')

try:
    key = kms.describe_key(KeyId=KMS_ALIAS)['KeyMetadata']
    key_id = key['KeyId']
    try:
        kms.delete_alias(AliasName=KMS_ALIAS)
        print(f'✓ KMS alias deleted: {KMS_ALIAS}')
    except ClientError as exc:
        if exc.response['Error']['Code'] != 'NotFoundException':
            raise
    if key.get('KeyState') != 'PendingDeletion':
        kms.schedule_key_deletion(KeyId=key_id, PendingWindowInDays=7)
        print(f'✓ KMS key scheduled for deletion in 7 days: {key_id}')
    else:
        print(f'✓ KMS key already pending deletion: {key_id}')
except ClientError as exc:
    if exc.response['Error']['Code'] != 'NotFoundException':
        raise
    print(f'✓ KMS key already absent: {KMS_ALIAS}')

oidc_provider_arn = globals().get('provider_arn')
if oidc_provider_arn is None:
    for item in iam.list_open_id_connect_providers().get('OpenIDConnectProviderList', []):
        details = iam.get_open_id_connect_provider(OpenIDConnectProviderArn=item['Arn'])
        if details['Url'].rstrip('/') == OIDC_HOSTPATH:
            oidc_provider_arn = item['Arn']
            break
if oidc_provider_arn:
    iam.delete_open_id_connect_provider(OpenIDConnectProviderArn=oidc_provider_arn)
    print(f'✓ IAM OIDC provider deleted: {oidc_provider_arn}')
else:
    print(f'✓ IAM OIDC provider already absent: {OIDC_ISSUER_URL}')

print('✓ Complete AWS cleanup finished')

✓ Deleted inline role policy: vault-kms-auto-unseal-oidc
✓ IAM role deleted: vault-kms-unseal-oidc
✓ KMS alias deleted: alias/vault-auto-unseal-oidc
✓ KMS key scheduled for deletion in 7 days: 234eabb2-5c2e-46bc-9603-98f0f9f17b78
✓ IAM OIDC provider deleted: arn:aws:iam::467008425114:oidc-provider/nondemanding-indistinguishably-claude.ngrok-free.dev
✓ Complete AWS cleanup finished
